# Multimodal Models 1: CLIP-Style Image–Text Alignment

## Case study: contrastive learning on CIFAR-10

CLIP learns a shared representation space for images and text. A vision encoder maps an image to a vector; a text encoder maps a caption to another vector. Training pulls matched image–caption pairs together and pushes mismatched pairs apart. Unlike a classifier trained only on fixed labels, the resulting similarity scores can be used to compare images with new candidate descriptions.

For a batch of $N$ pairs, the symmetric InfoNCE objective applies cross-entropy to the image-to-text and text-to-image similarity matrices. Similarities are cosine-normalized and scaled by a learned or fixed temperature. Each diagonal entry is the matching pair. CIFAR-10 captions here are generated from class labels, so this is a controlled educational setup rather than rich language supervision; class templates also mean the model may learn shortcuts.

The real CLIP uses large pretrained encoders and hundreds of millions of image–text pairs. This notebook trains small encoders on a small CIFAR-10 subset to expose the mechanics. It attempts to load CIFAR-10 through torchvision and falls back to simple colored icons if data or torchvision is unavailable.

## 1. Load paired image–text data

Each CIFAR-10 label is converted into one of a few caption templates. For the fallback, synthetic colored images stand in for the ten classes and preserve the same image/label interface. Training uses a small subset so the experiment remains bounded.

**What this cell does:** loads CIFAR-10 or the fallback dataset, selects examples, and prepares class names and caption templates.

In [ ]:
import random,numpy as np,torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset,DataLoader,Subset
SEED=17;random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
classes=['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
templates=['a photo of a {}','a small {} in a natural scene','an image showing a {}']
try:
 from torchvision.datasets import CIFAR10
 from torchvision import transforms
 base=CIFAR10('./data',train=True,download=True,transform=transforms.ToTensor())
except Exception as e:
 print('CIFAR-10 unavailable; using synthetic colored-icon fallback:',type(e).__name__)
 class Synthetic(Dataset):
  def __len__(self):return 1200
  def __getitem__(self,i):
   label=i%10; im=torch.rand(3,32,32)*.08; color=torch.zeros(3);color[label%3]=.7;color[(label+1)%3]=.2
   im[:,8:24,8:24]+=color[:,None,None];return im.clamp(0,1),label
 base=Synthetic()
ids=np.random.permutation(len(base))[:min(2000,len(base))]; data=Subset(base,ids)
print('device:',device,'examples:',len(data),'classes:',len(classes))


## 2. Tokenization, dual encoders, and projection space

The demo text encoder uses a tiny word vocabulary and mean-pooled embeddings. The vision encoder uses a small CNN. Projection heads map both modalities to the same embedding dimension, followed by L2 normalization. Real CLIP uses a Transformer text encoder and a stronger image backbone.

**What this cell does:** builds captions, tokenizes words, and defines the two encoders with a shared embedding size.

In [ ]:
vocab=['<pad>','a','photo','of','small','in','natural','scene','an','image','showing']+classes
stoi={w:i for i,w in enumerate(vocab)}
def tokenize(label):
 words=templates[label%len(templates)].format(classes[label]).split(); ids=[stoi.get(w,0) for w in words]
 return ids+[0]*(8-len(ids))
class PairData(Dataset):
 def __len__(self):return len(data)
 def __getitem__(self,i):
  im,y=data[i];return im,y,torch.tensor(tokenize(int(y)))
loader=DataLoader(PairData(),batch_size=64,shuffle=True,drop_last=True)
class CLIPMini(nn.Module):
 def __init__(self):
  super().__init__();self.vision=nn.Sequential(nn.Conv2d(3,32,3,2,1),nn.ReLU(),nn.Conv2d(32,64,3,2,1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(64,128))
  self.word=nn.Embedding(len(vocab),64,padding_idx=0);self.text=nn.Sequential(nn.Linear(64,128),nn.GELU(),nn.Linear(128,128));self.logit_scale=nn.Parameter(torch.tensor(2.3))
 def forward(self,im,tok):
  a=F.normalize(self.vision(im),dim=-1); emb=self.word(tok); mask=(tok!=0).unsqueeze(-1); pooled=(emb*mask).sum(1)/mask.sum(1).clamp_min(1)
  b=F.normalize(self.text(pooled),dim=-1);return a,b
model=CLIPMini().to(device);print('parameters:',sum(p.numel() for p in model.parameters()))


## 3. Symmetric contrastive objective and training

For normalized image features $v$ and text features $t$, logits are $s v t^T$, where $s$ is the inverse temperature. The target for row and column $i$ is pair index $i$. Averaging both cross-entropies encourages retrieval in both directions.

**What this cell does:** trains the dual encoder for a few epochs and reports in-batch retrieval accuracy. This metric is measured on training batches, so it is not a held-out generalization estimate.

In [ ]:
opt=torch.optim.AdamW(model.parameters(),lr=2e-3);EPOCHS=5;losses=[]
for ep in range(EPOCHS):
 model.train();ls=[];acc=[]
 for im,y,tok in loader:
  im,tok=im.to(device),tok.to(device);v,t=model(im,tok); logits=model.logit_scale.exp().clamp(max=100)*(v@t.T); target=torch.arange(len(im),device=device)
  loss=(F.cross_entropy(logits,target)+F.cross_entropy(logits.T,target))/2
  opt.zero_grad(set_to_none=True);loss.backward();opt.step();ls.append(loss.item());acc.append((logits.argmax(1)==target).float().mean().item())
 losses.append(np.mean(ls));print(f'epoch {ep+1}: contrastive loss={np.mean(ls):.3f}, in-batch retrieval={np.mean(acc):.2f}')
plt.plot(losses,marker='o');plt.xlabel('epoch');plt.ylabel('loss');plt.grid();plt.show()


## 4. Zero-shot-style classification and retrieval

To classify an image, compare its embedding with candidate text embeddings and select the highest cosine similarity. This is zero-shot in the sense that the decision is made through text prompts rather than a trained softmax classifier, but this small model has seen the same class names during contrastive training.

**What this cell does:** embeds candidate prompts, scores a batch of images, and displays predicted labels and similarity values.

In [ ]:
model.eval();im,y,tok=next(iter(loader));im=im.to(device)
with torch.no_grad():
 prompts=torch.stack([torch.tensor(tokenize(i)) for i in range(10)]).to(device);v,_=model(im,torch.zeros(len(im),8,dtype=torch.long,device=device));_,txt=model(torch.zeros(10,3,32,32,device=device),prompts);scores=v@txt.T;pred=scores.argmax(1)
fig,ax=plt.subplots(1,6,figsize=(12,2.5))
for i,a in enumerate(ax):a.imshow(im[i].cpu().permute(1,2,0));a.set_title(f"true {classes[int(y[i])]}/pred {classes[int(pred[i])]}",fontsize=8);a.axis('off')
plt.tight_layout();plt.show();print('candidate scores for first image:',[(classes[i],round(float(scores[0,i]),3)) for i in range(10)])


## Key takeaways

- CLIP aligns image and text embeddings using matched pairs and a symmetric contrastive objective.
- Temperature controls the sharpness of similarity logits; both retrieval directions contribute to training.
- Prompt-based classification reuses text embeddings instead of a fixed classifier head.
- Label-template captions are weak supervision and do not represent the diversity of natural language.
- A small model trained from scratch is a mechanics demo, not a reproduction of pretrained CLIP.

**References:** [CLIP paper](https://arxiv.org/abs/2103.00020).